In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H10b — BSDT Gravity V2 FAST: UDP-style Physics for Hard SAT
# ══════════════════════════════════════════════════════════════════════
# Key speed fixes vs slow H10b:
#   - Gravity applied every 20 steps (not every step) → 20× fewer cdist
#   - AMP float16 for energy computation
#   - torch.compile on energy function
#   - Cached nearest-elite targets (reused between gravity updates)
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time
from numba import njit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))

# ── Numba WalkSAT (flat-array adjacency) ─────────────────────────────
@njit(cache=True)
def walksat_numba(clauses_arr, assignment, max_flips=100000, p=0.4):
    m = clauses_arr.shape[0]
    k = 3
    n = assignment.shape[0]
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(k):
            var_count[abs(clauses_arr[c, j]) - 1] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for v in range(n):
        var_off[v + 1] = var_off[v] + var_count[v]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(k):
            v = abs(clauses_arr[c, j]) - 1
            var_adj[var_off[v] + fill[v]] = c
            fill[v] += 1
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(k):
            v = abs(clauses_arr[c, j]) - 1
            s = 1 if clauses_arr[c, j] > 0 else 0
            if assignment[v] == s:
                clause_sat[c] += 1
    for flip in range(max_flips):
        n_unsat = 0
        for c in range(m):
            if clause_sat[c] == 0:
                n_unsat += 1
        if n_unsat == 0:
            return assignment, flip
        target = np.random.randint(n_unsat)
        ci = -1; cnt = 0
        for c in range(m):
            if clause_sat[c] == 0:
                if cnt == target:
                    ci = c; break
                cnt += 1
        if np.random.random() < p:
            v = abs(clauses_arr[ci, np.random.randint(k)]) - 1
        else:
            best_v = abs(clauses_arr[ci, 0]) - 1
            best_brk = 999999
            for j in range(k):
                vc = abs(clauses_arr[ci, j]) - 1
                brk = 0
                for idx in range(var_off[vc], var_off[vc + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 1:
                        for jj in range(k):
                            vv = abs(clauses_arr[cc, jj]) - 1
                            ss = 1 if clauses_arr[cc, jj] > 0 else 0
                            if vv == vc and assignment[vv] == ss:
                                brk += 1; break
                if brk < best_brk:
                    best_brk = brk; best_v = vc
            v = best_v
        assignment[v] = 1 - assignment[v]
        for idx in range(var_off[v], var_off[v + 1]):
            cc = var_adj[idx]
            clause_sat[cc] = 0
            for jj in range(k):
                vv = abs(clauses_arr[cc, jj]) - 1
                ss = 1 if clauses_arr[cc, jj] > 0 else 0
                if assignment[vv] == ss:
                    clause_sat[cc] += 1
    return assignment, max_flips

# ══════════════════════════════════════════════════════════════════════
#  BSDTGravityV2 — FAST version
# ══════════════════════════════════════════════════════════════════════
class BSDTGravityV2:
    def __init__(self, n, clauses, mu_scale=0.1,
                 G_max=0.10, top_k_frac=0.1, gravity_start=0.2,
                 elite_repulsion=0.5, gravity_interval=20):
        self.n = n
        self.m = len(clauses)
        self.mu_scale = mu_scale
        self.G_max = G_max
        self.top_k_frac = top_k_frac
        self.gravity_start = gravity_start
        self.elite_repulsion = elite_repulsion
        self.gravity_interval = gravity_interval  # apply gravity every N steps
        self.device = device

        # Pre-compute clause tensors on GPU
        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t  = torch.tensor(vs_list, dtype=torch.long,    device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()

        # Pre-compute WalkSAT clause format
        vars_np  = np.array(vs_list, dtype=np.int32)
        signs_np = np.array(ss_list, dtype=np.int32)
        self.clauses_ws = (vars_np + 1) * signs_np

    def _energy_core(self, s, mu_val, vars_t, signs_t):
        """Compiled energy function — AMP-compatible."""
        lit = s[:, vars_t] * signs_t.unsqueeze(0)
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum(-1)
        if mu_val > 0:
            return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(-1)
        return e_sat

    def find_best_particle(self, s):
        """Vectorised GPU SAT check — returns (best_idx, violations)."""
        with torch.no_grad():
            x = (s > 0).long()
            lit_ok = (x[:, self.vars_t] == self.pos_mask.unsqueeze(0))
            n_sat = lit_ok.any(dim=2).sum(dim=1)
            best = n_sat.argmax()
            return best.item(), self.m - n_sat[best].item()

    def check_single(self, x):
        with torch.no_grad():
            n_sat = int((x[self.vars_t] == self.pos_mask).any(dim=1).sum())
            return n_sat == self.m, self.m - n_sat

    def gravity_flow(self, steps=4000, particles=1000,
                     schedule='delay70', lr=0.02):
        n = self.n
        gi = self.gravity_interval
        s = torch.randn(particles, n, device=self.device) * 0.1
        s.requires_grad_(True)

        grav_step  = int(self.gravity_start * steps)
        delay_step = int(0.7 * steps)
        top_k = max(1, int(self.top_k_frac * particles))
        theta = None
        use_amp = (self.device.type == 'cuda')

        # Cached gravity targets (refreshed every gi steps)
        cached_target = None

        for step in range(steps):
            # ── μ schedule ──
            if step < delay_step:
                mu = 0.0
            else:
                t_l = (step - delay_step) / (steps - delay_step)
                mu = self.mu_scale * 0.5 * (1.0 - np.cos(np.pi * t_l))

            # ── forward + backward (with AMP) ──
            if use_amp:
                with torch.amp.autocast('cuda'):
                    e = self._energy_core(s, mu, self.vars_t, self.signs_t)
                    e_f32 = e.float()
            else:
                e_f32 = self._energy_core(s, mu, self.vars_t, self.signs_t)

            e_vals = e_f32.detach()
            e_f32.sum().backward()

            with torch.no_grad():
                # ── Adaptive damping ──
                if theta is None:
                    theta = float(e_vals.median()) + 1e-8
                damp = 1.0 / (1.0 + e_vals.unsqueeze(1) / theta)
                s.sub_(lr * damp * s.grad)

                # ── Gravity (every gi steps only) ──
                if step >= grav_step and (step - grav_step) % gi == 0:
                    progress = (step - grav_step) / (steps - grav_step)
                    # Scale g by gi to compensate for fewer applications
                    g = self.G_max * progress * progress * gi

                    _, top_idx = e_vals.topk(top_k, largest=False)
                    elite = s[top_idx]

                    # Nearest-elite targets
                    d = torch.cdist(s, elite)
                    cached_target = elite[d.argmin(dim=1)]

                    # Elite mutual repulsion
                    if top_k > 1:
                        ed = d[top_idx]
                        ed.fill_diagonal_(float('inf'))
                        nn_e = ed.argmin(dim=1)
                        push = elite - elite[nn_e]
                        pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                        s[top_idx] += (self.elite_repulsion * g) * (push / pn)

                    s.add_(g * (cached_target - s))

                elif step >= grav_step and cached_target is not None:
                    # Reuse cached targets between gravity updates
                    progress = (step - grav_step) / (steps - grav_step)
                    g = self.G_max * progress * progress
                    s.add_(g * (cached_target - s))

                s.clamp_(-1, 1)

                if (step + 1) % 200 == 0:
                    theta = float(e_vals.median()) + 1e-8

            s.requires_grad_(True)
            if s.grad is not None:
                s.grad.zero_()

        return s.detach()

# Try to compile the energy function for extra speed
try:
    BSDTGravityV2._energy_core = torch.compile(BSDTGravityV2._energy_core)
    print('✓ torch.compile applied to energy function')
except Exception:
    print('⚠ torch.compile unavailable — using eager mode')

# Numba warmup
_ = walksat_numba(np.array([[1, -2, 3]], dtype=np.int32),
                  np.array([1, 0, 1], dtype=np.int32), max_flips=10)

print('✓ BSDTGravityV2 FAST loaded')
print(f'  Device: {device}')
if device.type == 'cuda':
    print(f'  GPU:    {torch.cuda.get_device_name()}')
print(f'  Gravity: every 20 steps, AMP float16, cached targets')
print(f'  Physics: G_max=0.10, start=20%, repulsion=0.5, damping=adaptive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H10b Experiment: Gravity V2 on hard α = {3.8, 4.0, 4.2}
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50
PARTICLES = 1000
STEPS    = 4000

# H9b baseline
baseline = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}

results = {}
print('=' * 84)
print('H10b GRAVITY V2 FAST — nearest-elite + repulsion + adaptive damping')
print('=' * 84)
print(f"  {'α':>5} | {'n':>5} | {'S1':>6} | {'Final':>6} | {'Viols':>5} | "
      f"{'Flips':>7} | {'WS':>7} | {'Base':>5} | {'Δ':>6} | Time")
print('  ' + '-' * 78)

for alpha in ALPHAS:
    for n_var in NS:
        m = int(alpha * n_var)
        t0 = time.time()
        s1 = ws = wt = 0
        tot_v = tot_f = fc = 0

        for inst in range(N_INST):
            clauses = generate_3sat_instance(n_var, m)
            eng = BSDTGravityV2(n_var, clauses)
            sf = eng.gravity_flow(steps=STEPS, particles=PARTICLES)

            best_idx, viols = eng.find_best_particle(sf)

            if viols == 0:
                s1 += 1
            else:
                tot_v += viols; fc += 1
                x_np = (sf[best_idx] > 0).cpu().numpy().astype(np.int32)
                sol, flips = walksat_numba(eng.clauses_ws, x_np.copy(),
                                          max_flips=100000, p=0.4)
                tot_f += flips; wt += 1
                sol_t = torch.tensor(sol, dtype=torch.long, device=device)
                sat, _ = eng.check_single(sol_t)
                if sat:
                    ws += 1

            # Progress indicator every 10 instances
            if (inst + 1) % 10 == 0:
                elapsed_so_far = time.time() - t0
                print(f'    α={alpha}, n={n_var}: {inst+1}/{N_INST} '
                      f'({elapsed_so_far:.0f}s, '
                      f'~{elapsed_so_far/(inst+1)*N_INST:.0f}s total)')

        elapsed = time.time() - t0
        final = (s1 + ws) / N_INST * 100
        avg_v = tot_v / max(fc, 1)
        avg_f = tot_f // max(wt, 1)
        base = baseline[(alpha, n_var)]
        delta = final - base
        tag = '★' if final >= 95 else ('▲' if delta > 0 else
              ('=' if delta == 0 else '▼'))

        results[(alpha, n_var)] = {
            's1': s1 / N_INST * 100, 'final': final,
            'viols': avg_v, 'flips': avg_f,
            'ws': f'{ws}/{wt}', 'base': base, 'delta': delta
        }
        print(f'  {alpha:5.1f} | {n_var:5d} | {s1/N_INST*100:5.1f}% | '
              f'{final:5.1f}% | {avg_v:5.1f} | {avg_f:7d} | '
              f'{ws:3d}/{wt:<3d} | {base:4.0f}% | {delta:+5.1f}% | '
              f'{elapsed:4.0f}s {tag}')
    print('  ' + '-' * 78)

print('\nGenerating charts...\n')

# ── Charts ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, n_var in enumerate(NS):
    ax = axes[i]
    bv = [baseline[(a, n_var)] for a in ALPHAS]
    gv = [results[(a, n_var)]['final'] for a in ALPHAS]
    x = range(len(ALPHAS))
    ax.bar([xi - 0.15 for xi in x], bv, 0.3,
           label='H9b baseline', color='#e74c3c', alpha=0.8)
    ax.bar([xi + 0.15 for xi in x], gv, 0.3,
           label='H10b gravity V2', color='#2ecc71', alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for xi, (b, g2) in enumerate(zip(bv, gv)):
        d = g2 - b
        if d != 0:
            ax.annotate(f'{d:+.0f}%', (xi + 0.15, g2 + 2), ha='center',
                        fontsize=9, fontweight='bold',
                        color='green' if d > 0 else 'red')

fig.suptitle('H10b: Gravity V2 vs H9b Baseline — Hard α Region',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('h10b_gravity_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h10b_gravity_v2.png')